# 🗺️ Cartographie des Points Noirs — Réseau Ferroviaire Belge
**Projet Data Science — Tahar GUENFOUD**  
**Source :** Open Data Infrabel  
**Stack :** Python · Pandas · Folium · Plotly  

---
## 📋 Plan du notebook
1. Installation des dépendances
2. Téléchargement des données Infrabel
3. Exploration des données brutes
4. Nettoyage et transformation
5. Agrégation par gare
6. Cartographie interactive (Folium)
7. Analyse graphique (Plotly)
8. Conclusions & insights

---
## ⚙️ Étape 0 — Installation des dépendances

In [1]:
# Installation des librairies nécessaires
# À exécuter une seule fois
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'pandas', 'folium', 'plotly', 'requests', 'openpyxl',
                '--quiet'])
print('✅ Librairies installées')

✅ Librairies installées


---
## 📦 Imports

In [2]:
import requests
import pandas as pd
import folium
from folium.plugins import HeatMap
import plotly.express as px
import plotly.graph_objects as go
from io import StringIO
import os
import warnings
warnings.filterwarnings('ignore')

# Dossiers de sortie
os.makedirs('../data/raw',   exist_ok=True)
os.makedirs('../data/clean', exist_ok=True)

print('✅ Imports OK')

✅ Imports OK


---
## 1️⃣ Téléchargement des données Infrabel

In [3]:
# --- Récupération de l'index des fichiers mensuels ---
INDEX_URL = (
    'https://opendata.infrabel.be/api/explore/v2.1/catalog/datasets/'
    'stiptheid-gegevens-maandelijksebestanden/records?limit=100&lang=fr'
)

print('📋 Récupération de l\'index...')
resp = requests.get(INDEX_URL, timeout=30)
df_index = pd.DataFrame(resp.json()['results'])
df_index = df_index.sort_values('mois', ascending=False).reset_index(drop=True)

print(f'   → {len(df_index)} mois disponibles')
print(f'   → Période : {df_index["mois"].iloc[-1]} → {df_index["mois"].iloc[0]}')
df_index.head()

📋 Récupération de l'index...
   → 100 mois disponibles
   → Période : 2014-03 → 2026-02


,mois,link_to_data
0,2026-02,https://fr.ftp.opendatasoft.com/infrabel/Punct...
1,2026-01,https://fr.ftp.opendatasoft.com/infrabel/Punct...
2,2025-12,https://fr.ftp.opendatasoft.com/infrabel/Punct...
3,2025-11,https://fr.ftp.opendatasoft.com/infrabel/Punct...
4,2025-10,https://fr.ftp.opendatasoft.com/infrabel/Punct...


In [4]:
# --- Téléchargement des 12 derniers mois ---
NB_MOIS = 12
df_recent = df_index.head(NB_MOIS)

print(f'🚄 Téléchargement des {NB_MOIS} derniers mois...')
print(f'   Période : {df_recent["mois"].iloc[-1]} → {df_recent["mois"].iloc[0]}\n')

all_dfs = []

for _, row in df_recent.iterrows():
    mois = row['mois']
    url  = row['link_to_data']
    try:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        df_month = pd.read_csv(StringIO(r.text), sep=',', encoding='utf-8')
        df_month['mois'] = mois
        all_dfs.append(df_month)
        print(f'   ✅ {mois} — {len(df_month):,} lignes | colonnes : {list(df_month.columns)[:4]}...')
    except Exception as e:
        print(f'   ⚠️  {mois} — Erreur : {e}')

# Fusion de tous les mois
df_raw = pd.concat(all_dfs, ignore_index=True)

# Sauvegarde brute
df_raw.to_csv('../data/raw/ponctualite_merged.csv', index=False, encoding='utf-8-sig')

print(f'\n📊 Total : {len(df_raw):,} lignes | {len(df_raw.columns)} colonnes')
print(f'💾 Sauvegardé : data/raw/ponctualite_merged.csv')

🚄 Téléchargement des 12 derniers mois...
   Période : 2025-01 → 2026-02

   ✅ 2026-02 — 1,897,533 lignes | colonnes : ['DATDEP', 'CIRC_TYP', 'TRAIN_NO', 'RELATION']...
   ✅ 2026-01 — 1,889,339 lignes | colonnes : ['DATDEP', 'CIRC_TYP', 'TRAIN_NO', 'RELATION']...
   ✅ 2025-12 — 2,066,672 lignes | colonnes : ['DATDEP', 'CIRC_TYP', 'TRAIN_NO', 'RELATION']...
   ✅ 2025-11 — 1,757,601 lignes | colonnes : ['DATDEP', 'CIRC_TYP', 'TRAIN_NO', 'RELATION']...
   ✅ 2025-10 — 2,060,066 lignes | colonnes : ['DATDEP', 'CIRC_TYP', 'TRAIN_NO', 'RELATION']...
   ✅ 2025-09 — 2,015,606 lignes | colonnes : ['DATDEP', 'CIRC_TYP', 'TRAIN_NO', 'RELATION']...
   ✅ 2025-08 — 1,861,871 lignes | colonnes : ['DATDEP', 'CIRC_TYP', 'TRAIN_NO', 'RELATION']...
   ✅ 2025-07 — 1,916,850 lignes | colonnes : ['DATDEP', 'TRAIN_NO', 'RELATION', 'TRAIN_SERV']...
   ✅ 2025-05 — 1,919,566 lignes | colonnes : ['DATDEP', 'TRAIN_NO', 'RELATION', 'TRAIN_SERV']...
   ✅ 2025-04 — 1,770,587 lignes | colonnes : ['DATDEP', 'TRAIN_NO', 

---
## 2️⃣ Exploration des données brutes

In [5]:
df_raw.columns
   

Index(['DATDEP', 'CIRC_TYP', 'TRAIN_NO', 'RELATION', 'TRAIN_SERV', 'OP1_COD',
       'THOP1_COD', 'PTCAR_NO', 'PTCAR_LG_NM_NL', 'LINE_NO_DEP',
       'REAL_DATE_ARR', 'REAL_TIME_ARR', 'REAL_DATE_DEP', 'REAL_TIME_DEP',
       'PLANNED_DATE_ARR', 'PLANNED_TIME_ARR', 'PLANNED_TIME_DEP',
       'PLANNED_DATE_DEP', 'DELAY_ARR', 'DELAY_DEP', 'RELATION_DIRECTION',
       'LINE_NO_ARR', 'mois'],
      dtype='object')

In [6]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22642916 entries, 0 to 22642915
Data columns (total 23 columns):
 #   Column              Dtype  
---  ------              -----  
 0   DATDEP              object 
 1   CIRC_TYP            int64  
 2   TRAIN_NO            int64  
 3   RELATION            object 
 4   TRAIN_SERV          object 
 5   OP1_COD             object 
 6   THOP1_COD           object 
 7   PTCAR_NO            int64  
 8   PTCAR_LG_NM_NL      object 
 9   LINE_NO_DEP         object 
 10  REAL_DATE_ARR       object 
 11  REAL_TIME_ARR       object 
 12  REAL_DATE_DEP       object 
 13  REAL_TIME_DEP       object 
 14  PLANNED_DATE_ARR    object 
 15  PLANNED_TIME_ARR    object 
 16  PLANNED_TIME_DEP    object 
 17  PLANNED_DATE_DEP    object 
 18  DELAY_ARR           float64
 19  DELAY_DEP           float64
 20  RELATION_DIRECTION  object 
 21  LINE_NO_ARR         object 
 22  mois                object 
dtypes: float64(2), int64(3), object(18)
memory usage: 3.9+ 

In [7]:
df_raw.describe()

,CIRC_TYP,TRAIN_NO,PTCAR_NO,DELAY_ARR,DELAY_DEP
count,22642916.0,2.264292e+07,2.264292e+07,2.153462e+07,2.153612e+07
mean,1.0,3.812871e+03,6.445665e+02,1.163590e+02,1.205754e+02
std,0.0,2.886824e+03,4.219555e+02,3.490171e+02,3.396621e+02
min,1.0,3.000000e+00,6.000000e+00,-8.578900e+04,-8.548200e+04
25%,1.0,2.088000e+03,2.590000e+02,-6.000000e+00,7.000000e+00
50%,1.0,3.255000e+03,6.140000e+02,3.700000e+01,3.800000e+01
75%,1.0,4.588000e+03,9.250000e+02,1.230000e+02,1.210000e+02
max,1.0,1.997900e+04,2.300000e+03,3.867700e+04,3.867700e+04


In [8]:
# Valeurs manquantes
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
pd.DataFrame({'Manquants': missing, 'Pct (%)': missing_pct})

,Manquants,Pct (%)
DATDEP,0,0.0
CIRC_TYP,0,0.0
TRAIN_NO,0,0.0
RELATION,0,0.0
TRAIN_SERV,0,0.0
OP1_COD,12086986,53.4
THOP1_COD,2243652,9.9
PTCAR_NO,0,0.0
PTCAR_LG_NM_NL,0,0.0
LINE_NO_DEP,1106960,4.9


In [9]:
# Aperçu des données
df_raw.head(5)

,DATDEP,CIRC_TYP,TRAIN_NO,RELATION,TRAIN_SERV,OP1_COD,THOP1_COD,PTCAR_NO,PTCAR_LG_NM_NL,LINE_NO_DEP,...,REAL_TIME_DEP,PLANNED_DATE_ARR,PLANNED_TIME_ARR,PLANNED_TIME_DEP,PLANNED_DATE_DEP,DELAY_ARR,DELAY_DEP,RELATION_DIRECTION,LINE_NO_ARR,mois
0,01FEB2026,1,2623,IC 08,SNCB/NMBS,NaN,NaN,810,MECHELEN,25,...,0:33:34,NaN,NaN,0:33:00,01FEB2026,NaN,34.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,NaN,2026-02
1,01FEB2026,1,2623,IC 08,SNCB/NMBS,=,=,219,BRUSSELS AIRPORT - ZAVENTEM,36C,...,0:46:38,01FEB2026,0:44:00,0:47:00,01FEB2026,29.0,-22.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36C,2026-02
2,01FEB2026,1,2623,IC 08,SNCB/NMBS,D,D,916,NOSSEGEM,36,...,0:50:14,01FEB2026,0:50:00,0:50:00,01FEB2026,14.0,14.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
3,01FEB2026,1,2623,IC 08,SNCB/NMBS,D,D,648,KORTENBERG,36,...,0:51:40,01FEB2026,0:52:00,0:52:00,01FEB2026,-20.0,-20.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
4,01FEB2026,1,2623,IC 08,SNCB/NMBS,D,D,33,KORTENBERG-GOEDEREN,36,...,0:51:47,01FEB2026,0:52:00,0:52:00,01FEB2026,-13.0,-13.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02


---
## 3️⃣ Nettoyage et transformation

In [10]:
df = df_raw.copy()

# --- Affichage des colonnes disponibles pour adapter le nettoyage ---
print('Colonnes disponibles :', list(df.columns))

# --- Normalisation des noms de colonnes (minuscules, sans espaces) ---
df.columns = (
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(' ', '_')
    .str.replace('(', '')
    .str.replace(')', '')
)
print('\nColonnes normalisées :', list(df.columns))

Colonnes disponibles : ['DATDEP', 'CIRC_TYP', 'TRAIN_NO', 'RELATION', 'TRAIN_SERV', 'OP1_COD', 'THOP1_COD', 'PTCAR_NO', 'PTCAR_LG_NM_NL', 'LINE_NO_DEP', 'REAL_DATE_ARR', 'REAL_TIME_ARR', 'REAL_DATE_DEP', 'REAL_TIME_DEP', 'PLANNED_DATE_ARR', 'PLANNED_TIME_ARR', 'PLANNED_TIME_DEP', 'PLANNED_DATE_DEP', 'DELAY_ARR', 'DELAY_DEP', 'RELATION_DIRECTION', 'LINE_NO_ARR', 'mois']

Colonnes normalisées : ['datdep', 'circ_typ', 'train_no', 'relation', 'train_serv', 'op1_cod', 'thop1_cod', 'ptcar_no', 'ptcar_lg_nm_nl', 'line_no_dep', 'real_date_arr', 'real_time_arr', 'real_date_dep', 'real_time_dep', 'planned_date_arr', 'planned_time_arr', 'planned_time_dep', 'planned_date_dep', 'delay_arr', 'delay_dep', 'relation_direction', 'line_no_arr', 'mois']


In [11]:
# --- Identification automatique des colonnes clés ---
col_gare    = None
col_retard  = None
col_total   = None
col_annule  = None
col_date    = None

for col in df.columns:
    c = col.lower()
    if any(x in c for x in ['station', 'gare', 'halte', 'stop']):
        col_gare = col
    elif any(x in c for x in ['delay', 'retard', 'vertraging']):
        col_retard = col
    elif any(x in c for x in ['total', 'aantal', 'count']) and 'train' in c:
        col_total = col
    elif any(x in c for x in ['cancel', 'annul', 'geannul']):
        col_annule = col
    elif any(x in c for x in ['date', 'maand', 'month']):
        col_date = col

print(f'Colonne gare    : {col_gare}')
print(f'Colonne retard  : {col_retard}')
print(f'Colonne total   : {col_total}')
print(f'Colonne annulé  : {col_annule}')
print(f'Colonne date    : {col_date}')

Colonne gare    : None
Colonne retard  : delay_dep
Colonne total   : None
Colonne annulé  : None
Colonne date    : planned_date_dep


In [12]:
# --- Nettoyage ---

# 1. Supprimer les doublons
avant = len(df)
df = df.drop_duplicates()
print(f'Doublons supprimés : {avant - len(df)}')

# 2. Convertir la colonne de retard en numérique
if col_retard:
    df[col_retard] = pd.to_numeric(df[col_retard], errors='coerce')
    # Convertir secondes → minutes si les valeurs sont grandes
    if df[col_retard].median() > 300:
        df[col_retard] = df[col_retard] / 60
        print(f'Colonne {col_retard} convertie : secondes → minutes')

# 3. Convertir la date
if col_date:
    df[col_date] = pd.to_datetime(df[col_date], errors='coerce')

# 4. Nettoyer les noms de gares
if col_gare:
    df[col_gare] = df[col_gare].astype(str).str.strip().str.title()
    df = df[df[col_gare] != 'Nan']

# Sauvegarde
df.to_csv('../data/clean/ponctualite_clean.csv', index=False, encoding='utf-8-sig')

print(f'\n✅ Données propres : {len(df):,} lignes')
print(f'💾 Sauvegardé : data/clean/ponctualite_clean.csv')
df.head()

Doublons supprimés : 0

✅ Données propres : 22,642,916 lignes
💾 Sauvegardé : data/clean/ponctualite_clean.csv


,datdep,circ_typ,train_no,relation,train_serv,op1_cod,thop1_cod,ptcar_no,ptcar_lg_nm_nl,line_no_dep,...,real_time_dep,planned_date_arr,planned_time_arr,planned_time_dep,planned_date_dep,delay_arr,delay_dep,relation_direction,line_no_arr,mois
0,01FEB2026,1,2623,IC 08,SNCB/NMBS,NaN,NaN,810,MECHELEN,25,...,0:33:34,NaN,NaN,0:33:00,2026-02-01,NaN,34.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,NaN,2026-02
1,01FEB2026,1,2623,IC 08,SNCB/NMBS,=,=,219,BRUSSELS AIRPORT - ZAVENTEM,36C,...,0:46:38,01FEB2026,0:44:00,0:47:00,2026-02-01,29.0,-22.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36C,2026-02
2,01FEB2026,1,2623,IC 08,SNCB/NMBS,D,D,916,NOSSEGEM,36,...,0:50:14,01FEB2026,0:50:00,0:50:00,2026-02-01,14.0,14.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
3,01FEB2026,1,2623,IC 08,SNCB/NMBS,D,D,648,KORTENBERG,36,...,0:51:40,01FEB2026,0:52:00,0:52:00,2026-02-01,-20.0,-20.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
4,01FEB2026,1,2623,IC 08,SNCB/NMBS,D,D,33,KORTENBERG-GOEDEREN,36,...,0:51:47,01FEB2026,0:52:00,0:52:00,2026-02-01,-13.0,-13.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02


---
## 4️⃣ Agrégation par gare

In [13]:
# --- Agrégation par gare ---
agg_dict = {}
if col_retard : agg_dict[col_retard] = 'mean'
if col_total  : agg_dict[col_total]  = 'sum'
if col_annule : agg_dict[col_annule] = 'sum'

if col_gare and agg_dict:
    df_gare = df.groupby(col_gare).agg(agg_dict).reset_index()
    df_gare.columns = ['gare'] + [c.replace(col_retard or '', 'retard_moyen_min')
                                      .replace(col_total  or '', 'total_trains')
                                      .replace(col_annule or '', 'total_annules')
                                  for c in df_gare.columns[1:]]

    # Trier par retard décroissant
    df_gare = df_gare.sort_values('retard_moyen_min', ascending=False).reset_index(drop=True)

    # Sauvegarde
    df_gare.to_csv('../data/clean/ponctualite_par_gare.csv', index=False, encoding='utf-8-sig')

    print(f'✅ {len(df_gare)} gares agrégées')
    print(f'💾 Sauvegardé : data/clean/ponctualite_par_gare.csv')
    print(f'\n🏆 Top 10 gares avec le plus de retards :')
    display(df_gare.head(10))
else:
    print('⚠️  Colonnes insuffisantes — affichage des données brutes')
    df_gare = df.copy()
    display(df_gare.head(10))

⚠️  Colonnes insuffisantes — affichage des données brutes


,datdep,circ_typ,train_no,relation,train_serv,op1_cod,thop1_cod,ptcar_no,ptcar_lg_nm_nl,line_no_dep,...,real_time_dep,planned_date_arr,planned_time_arr,planned_time_dep,planned_date_dep,delay_arr,delay_dep,relation_direction,line_no_arr,mois
0,01FEB2026,1,2623,IC 08,SNCB/NMBS,NaN,NaN,810,MECHELEN,25,...,0:33:34,NaN,NaN,0:33:00,2026-02-01,NaN,34.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,NaN,2026-02
1,01FEB2026,1,2623,IC 08,SNCB/NMBS,=,=,219,BRUSSELS AIRPORT - ZAVENTEM,36C,...,0:46:38,01FEB2026,0:44:00,0:47:00,2026-02-01,29.0,-22.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36C,2026-02
2,01FEB2026,1,2623,IC 08,SNCB/NMBS,D,D,916,NOSSEGEM,36,...,0:50:14,01FEB2026,0:50:00,0:50:00,2026-02-01,14.0,14.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
3,01FEB2026,1,2623,IC 08,SNCB/NMBS,D,D,648,KORTENBERG,36,...,0:51:40,01FEB2026,0:52:00,0:52:00,2026-02-01,-20.0,-20.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
4,01FEB2026,1,2623,IC 08,SNCB/NMBS,D,D,33,KORTENBERG-GOEDEREN,36,...,0:51:47,01FEB2026,0:52:00,0:52:00,2026-02-01,-13.0,-13.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
5,01FEB2026,1,2623,IC 08,SNCB/NMBS,D,D,368,ERPS-KWERPS,36,...,0:52:55,01FEB2026,0:53:00,0:53:00,2026-02-01,-5.0,-5.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
6,01FEB2026,1,2623,IC 08,SNCB/NMBS,D,D,1174,VELTEM,36,...,0:54:24,01FEB2026,0:55:00,0:55:00,2026-02-01,-36.0,-36.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
7,01FEB2026,1,2623,IC 08,SNCB/NMBS,D,D,553,HERENT,36,...,0:55:23,01FEB2026,0:56:00,0:56:00,2026-02-01,-37.0,-37.0,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
8,01FEB2026,1,2623,IC 08,SNCB/NMBS,NaN,NaN,715,LEUVEN,NaN,...,NaN,01FEB2026,1:00:00,NaN,NaT,-42.0,NaN,IC 08: ANTWERPEN-CENTRAAL -> HASSELT,36,2026-02
9,01FEB2026,1,3229,IC 26,SNCB/NMBS,NaN,NaN,319,DENDERMONDE,60,...,9:05:12,NaN,NaN,9:05:00,2026-02-01,NaN,12.0,IC 26: DENDERMONDE -> BRUSSEL-ZUID,NaN,2026-02


---
## 5️⃣ Coordonnées GPS des gares

In [14]:
# Coordonnées des principales gares belges
COORDS_GARES = {
    'Bruxelles-Central'   : (50.8453, 4.3571),
    'Bruxelles-Midi'      : (50.8354, 4.3363),
    'Bruxelles-Nord'      : (50.8590, 4.3613),
    'Bruxelles-Schuman'   : (50.8449, 4.3785),
    'Gand-Saint-Pierre'   : (51.0355, 3.7103),
    'Anvers-Central'      : (51.2172, 4.4211),
    'Liège-Guillemins'    : (50.6246, 5.5662),
    'Namur'               : (50.4671, 4.8681),
    'Charleroi-Sud'       : (50.4109, 4.4443),
    'Bruges'              : (51.1972, 3.2167),
    'Louvain'             : (50.8820, 4.7164),
    'Mons'                : (50.4540, 3.9520),
    'Tournai'             : (50.6050, 3.3880),
    'Hasselt'             : (50.9307, 5.3378),
    'Genk'                : (50.9659, 5.4985),
    'Courtrai'            : (50.8240, 3.2643),
    'Ottignies'           : (50.6680, 4.5680),
    'Mechelen'            : (51.0270, 4.4820),
    'Aalst'               : (50.9370, 4.0410),
    'Aarschot'            : (50.9860, 4.8290),
    'Dendermonde'         : (51.0270, 4.1010),
    'Leuven'              : (50.8820, 4.7164),
    'Antwerpen-Centraal'  : (51.2172, 4.4211),
    'Gent-Sint-Pieters'   : (51.0355, 3.7103),
    'Luik-Guillemins'     : (50.6246, 5.5662),
}

def get_coords(gare_name):
    """Cherche les coordonnées d'une gare par correspondance approx."""
    if pd.isna(gare_name):
        return None, None
    g = str(gare_name).strip()
    if g in COORDS_GARES:
        return COORDS_GARES[g]
    # Correspondance partielle
    for key, coords in COORDS_GARES.items():
        if g.lower() in key.lower() or key.lower() in g.lower():
            return coords
    return None, None

# Ajout des coordonnées
df_gare[['lat', 'lon']] = pd.DataFrame(
    df_gare['gare'].apply(get_coords).tolist(),
    index=df_gare.index
)

df_avec_coords = df_gare.dropna(subset=['lat', 'lon'])
print(f'✅ {len(df_avec_coords)} gares avec coordonnées GPS sur {len(df_gare)} total')
df_avec_coords.head()

KeyError: 'gare'

---
## 6️⃣ Cartographie interactive (Folium)

In [ ]:
# Colonne de valeur pour la carte
val_col = 'retard_moyen_min' if 'retard_moyen_min' in df_avec_coords.columns else df_avec_coords.columns[1]

# --- Création de la carte ---
carte = folium.Map(
    location=[50.5, 4.5],
    zoom_start=8,
    tiles='CartoDB positron'
)

# --- HeatMap ---
heat_data = [
    [row['lat'], row['lon'], float(row[val_col])]
    for _, row in df_avec_coords.iterrows()
    if pd.notna(row[val_col])
]
HeatMap(heat_data, radius=30, blur=20,
        gradient={'0.4': 'blue', '0.65': 'orange', '1': 'red'}).add_to(carte)

# --- Marqueurs colorés ---
q33 = df_avec_coords[val_col].quantile(0.33)
q66 = df_avec_coords[val_col].quantile(0.66)

for _, row in df_avec_coords.iterrows():
    val = row[val_col]
    if pd.isna(val): continue

    couleur = 'red' if val >= q66 else ('orange' if val >= q33 else 'green')
    niveau  = '🔴 Élevé' if val >= q66 else ('🟠 Modéré' if val >= q33 else '🟢 Bon')

    popup = f"""
    <b>🚉 {row['gare']}</b><br>
    Retard moyen : <b>{val:.1f} min</b><br>
    Niveau : {niveau}
    """
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=9,
        color=couleur, fill=True, fill_color=couleur, fill_opacity=0.8,
        popup=folium.Popup(popup, max_width=220),
        tooltip=f"🚉 {row['gare']} — {val:.1f} min"
    ).add_to(carte)

# Sauvegarde
os.makedirs('../app', exist_ok=True)
carte.save('../app/carte_points_noirs.html')

print(f'✅ Carte générée avec {len(heat_data)} points')
print('💾 Sauvegardée : app/carte_points_noirs.html')

# Affichage inline
carte

---
## 7️⃣ Analyse graphique (Plotly)

In [ ]:
# --- Top 15 gares les plus problématiques ---
top15 = df_gare.nlargest(15, val_col)

fig = px.bar(
    top15,
    x=val_col, y='gare',
    orientation='h',
    color=val_col,
    color_continuous_scale=['green', 'orange', 'red'],
    title='🏆 Top 15 — Gares avec le plus de retards moyens',
    labels={val_col: 'Retard moyen (min)', 'gare': 'Gare'}
)
fig.update_layout(height=500, yaxis={'categoryorder': 'total ascending'}, showlegend=False)
fig.show()

In [ ]:
# --- Évolution temporelle (si colonne date disponible) ---
if col_date and col_retard:
    df_trend = df.groupby(col_date)[col_retard].mean().reset_index()
    df_trend.columns = ['date', 'retard_moyen']

    fig2 = px.line(
        df_trend,
        x='date', y='retard_moyen',
        title='📈 Évolution du retard moyen dans le temps',
        labels={'retard_moyen': 'Retard moyen (min)', 'date': 'Date'},
        markers=True
    )
    fig2.update_traces(line_color='#e63946', line_width=2)
    fig2.show()
else:
    print('⚠️  Colonne date non disponible pour la tendance temporelle')

---
## 8️⃣ Conclusions & Insights

> À compléter après analyse des résultats

### 🔍 Observations clés
- **Gare la plus problématique :** ...
- **Retard moyen global :** ...
- **Tendance :** ...

### 💼 Formulation entretien (SNCB/Infrabel)
> *"J'ai analysé 12 mois de données Open Data Infrabel pour identifier les points noirs du réseau. J'ai découvert que [insight 1] et que [insight 2]. Cela suggère que [recommandation]."*

### 🚀 Prochaines étapes
- [ ] Ajouter plus de gares avec coordonnées GPS
- [ ] Intégrer les données météo (corrélation retards/météo)
- [ ] Construire un modèle prédictif (Projet 1 — Predictor)

In [15]:
!pwd

/mnt/c/Users/tahar/Documents/DataScience/projects/blackspots
